# EduSpace — Campus Facility Utilization Analysis
**Assessment:** U23AD491 — Data Science, DS Sprint Skill-Based Assessment 2  
**Dataset:** Synthetic records created for learning; not real campus observations.

This notebook runs the end-to-end workflow: load raw data, profile data quality, standardize facility names, clean and engineer features, analyze utilization, visualize trends, and export cleaned outputs and summary tables.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# If running in Google Colab, upload/extract the ZIP or set DATA_DIR to your extracted raw folder.
DATA_DIR = Path("../data/raw")
if not DATA_DIR.exists():
    DATA_DIR = Path("/content")  # convenient when the five CSV files are uploaded directly to Colab
OUT_DIR = Path("../data/cleaned")
CHART_DIR = Path("../output/charts")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

bookings_raw = pd.read_csv(DATA_DIR / "bookings.csv")
facilities_raw = pd.read_csv(DATA_DIR / "facilities.csv")
occupancy_raw = pd.read_csv(DATA_DIR / "occupancy.csv")
calendar_raw = pd.read_csv(DATA_DIR / "calendar.csv")
maintenance_raw = pd.read_csv(DATA_DIR / "maintenance.csv")

datasets_raw = {
    "Bookings": bookings_raw, "Facilities": facilities_raw,
    "Occupancy": occupancy_raw, "Calendar": calendar_raw,
    "Maintenance": maintenance_raw
}
print({name: df.shape for name, df in datasets_raw.items()})


In [ ]:
# Initial profile: types, missing values, exact duplicate rows
for name, df in datasets_raw.items():
    print(f"\\n--- {name} ---")
    display(df.head())
    print("Shape:", df.shape)
    print("Missing by column:\\n", df.isna().sum())
    print("Exact duplicate rows:", df.duplicated().sum())
    df.info()


In [ ]:
# Standardize known facility aliases consistently before joins/deduplication
facility_mapping = {
    "CR 101": "CR-101", "Classroom 101": "CR-101",
    "Lab-1": "LAB-1", "LAB 1": "LAB-1",
    "Library Main": "LIB-MAIN", "LIB MAIN": "LIB-MAIN",
    "Seminar Hall 1": "SEM-1", "SEM 1": "SEM-1"
}
bookings = bookings_raw.copy()
facilities = facilities_raw.copy()
occupancy = occupancy_raw.copy()
calendar = calendar_raw.copy()
maintenance = maintenance_raw.copy()

for df in [bookings, facilities, occupancy, maintenance]:
    df["facility"] = df["facility"].astype("string").str.strip().replace(facility_mapping)

# Parse date/time columns
bookings["date"] = pd.to_datetime(bookings["date"], errors="coerce")
bookings["start_td"] = pd.to_timedelta(bookings["start"] + ":00", errors="coerce")
bookings["end_td"] = pd.to_timedelta(bookings["end"] + ":00", errors="coerce")
bookings["booking_duration_hours"] = (bookings["end_td"] - bookings["start_td"]).dt.total_seconds()/3600

occupancy["timestamp"] = pd.to_datetime(occupancy["timestamp"], errors="coerce")
occupancy["count"] = pd.to_numeric(occupancy["count"], errors="coerce")
calendar["date"] = pd.to_datetime(calendar["date"], errors="coerce")
calendar["event"] = calendar["event"].fillna("No Event")
calendar["working_day"] = calendar["working_day"].astype("boolean")
calendar["exam_period"] = calendar["exam_period"].astype("boolean")
maintenance["downtime"] = pd.to_numeric(maintenance["downtime"], errors="coerce")
maintenance["issue"] = maintenance["issue"].fillna("Unknown Issue")

# Remove exact duplicate bookings only after standardizing aliases.
duplicate_booking_count = int(bookings.duplicated(subset=["facility","date","start_td","end_td","purpose"]).sum())
bookings = bookings.drop_duplicates(subset=["facility","date","start_td","end_td","purpose"]).copy()
bookings = bookings.loc[bookings["booking_duration_hours"] > 0].copy()
print("Duplicate booking rows removed:", duplicate_booking_count)
print("Bookings after deduplication:", bookings.shape)


In [ ]:
# Capacity validation and feature engineering
capacity_map = facilities.set_index("facility")["capacity"].to_dict()
occupancy["capacity"] = occupancy["facility"].map(capacity_map)
occupancy["over_capacity_flag"] = occupancy["count"] > occupancy["capacity"]
occupancy["utilization_pct"] = np.where(
    occupancy["count"].notna() & occupancy["capacity"].notna(),
    occupancy["count"] / occupancy["capacity"] * 100, np.nan
)
occupancy["date"] = occupancy["timestamp"].dt.normalize()
occupancy["hour"] = occupancy["timestamp"].dt.hour
occupancy["minute"] = occupancy["timestamp"].dt.minute
occupancy["time_slot"] = occupancy["timestamp"].dt.strftime("%H:%M")
occupancy = occupancy.merge(
    calendar[["date","working_day","event","exam_period"]],
    on="date", how="left", validate="many_to_one"
)
bookings["start_hour"] = bookings["start_td"].dt.total_seconds()/3600
bookings["day_of_week"] = bookings["date"].dt.day_name()
bookings = bookings.merge(
    calendar[["date","working_day","event","exam_period"]],
    on="date", how="left", validate="many_to_one"
)

print("Missing occupancy counts:", occupancy["count"].isna().sum())
print("Over-capacity readings flagged:", int(occupancy["over_capacity_flag"].sum()))
print("Unknown facility values in occupancy:", int(occupancy["capacity"].isna().sum()))


In [ ]:
# EDA summaries
valid_occ = occupancy.dropna(subset=["count","capacity"]).copy()
facility_summary = valid_occ.groupby("facility").agg(
    readings=("count","count"),
    average_occupancy=("count","mean"),
    peak_occupancy=("count","max"),
    average_utilization_pct=("utilization_pct","mean"),
    peak_utilization_pct=("utilization_pct","max"),
    over_capacity_readings=("over_capacity_flag","sum")
).reset_index().merge(facilities[["facility","type","capacity","block"]], on="facility", how="left")
facility_summary = facility_summary.sort_values("average_utilization_pct", ascending=False)
display(facility_summary.round(2))

peak_summary = valid_occ.groupby("hour").agg(
    average_occupancy=("count","mean"),
    average_utilization_pct=("utilization_pct","mean"),
    readings=("count","count")
).reset_index()
display(peak_summary.round(2))

exam_summary = valid_occ.groupby("exam_period").agg(
    average_occupancy=("count","mean"),
    average_utilization_pct=("utilization_pct","mean"),
    readings=("count","count")
).reset_index()
exam_summary["period"] = exam_summary["exam_period"].map({True:"Exam period",False:"Non-exam period"})
display(exam_summary.round(2))


In [ ]:
# Maintenance and booking summaries
maintenance_summary = maintenance.groupby("facility").agg(
    maintenance_records=("facility","size"),
    known_downtime_hours=("downtime","sum"),
    missing_downtime_records=("downtime", lambda s: s.isna().sum()),
    missing_issue_records=("issue", lambda s: s.eq("Unknown Issue").sum())
).reset_index().sort_values("known_downtime_hours", ascending=False)
display(maintenance_summary.round(2))

booking_summary = bookings.groupby("facility").agg(
    booking_count=("facility","size"),
    booked_hours=("booking_duration_hours","sum")
).reset_index().merge(facilities[["facility","type","capacity","block"]], on="facility", how="left")
display(booking_summary.round(2))


In [ ]:
# Visualizations
fig, ax = plt.subplots(figsize=(10,5))
facility_summary.sort_values("average_utilization_pct").plot(
    x="facility", y="average_utilization_pct", kind="barh", ax=ax, legend=False
)
ax.set_title("Average Occupancy Utilization by Facility")
ax.set_xlabel("Average occupancy / capacity (%)")
ax.set_ylabel("Facility")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9,4.5))
ax.plot(peak_summary["hour"], peak_summary["average_utilization_pct"], marker="o")
ax.set_title("Average Utilization by Hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Average utilization (%)")
ax.grid(True, alpha=.3)
plt.tight_layout()
plt.show()

exam_plot = exam_summary.copy()
exam_plot["period"] = exam_plot["exam_period"].map({True:"Exam period",False:"Non-exam period"})
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(exam_plot["period"], exam_plot["average_utilization_pct"])
ax.set_title("Exam vs Non-Exam Average Utilization")
ax.set_ylabel("Average utilization (%)")
plt.tight_layout()
plt.show()


In [ ]:
# Export cleaned data and analysis tables
bookings.to_csv(OUT_DIR / "bookings_cleaned.csv", index=False)
facilities.to_csv(OUT_DIR / "facilities_cleaned.csv", index=False)
occupancy.to_csv(OUT_DIR / "occupancy_cleaned.csv", index=False)
calendar.to_csv(OUT_DIR / "calendar_cleaned.csv", index=False)
maintenance.to_csv(OUT_DIR / "maintenance_cleaned.csv", index=False)
facility_summary.to_csv(OUT_DIR / "facility_utilization_summary.csv", index=False)
peak_summary.to_csv(OUT_DIR / "peak_time_summary.csv", index=False)
exam_summary.to_csv(OUT_DIR / "exam_period_summary.csv", index=False)
maintenance_summary.to_csv(OUT_DIR / "maintenance_summary.csv", index=False)
booking_summary.to_csv(OUT_DIR / "booking_summary.csv", index=False)
print("Export complete:", OUT_DIR.resolve())
